In [1]:
# ====== ПАРАМЕТРЫ ======
BASE_DIR   = "split_windows_512"   # папка с окнами
ANOMALY_N  = 3
NORMAL_N   = 3
SCORE_COL  = "manual_score"   # или "iso_score"

# ====== ЕСЛИ df_feat УЖЕ ЕСТЬ, ПРОПУСТИ ЭТО ======
# Если у тебя уже есть DataFrame df_feat с признаками и score — просто загрузи его из файла.
# Ниже краткая версия вычисления (без спектральных признаков), как раньше:

import os
import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest

def extract_features(df):
    feats = {}
    for ph in ['current_R', 'current_S', 'current_T']:
        feats[f'RMS_{ph[-1]}']  = np.sqrt(np.mean(df[ph]**2))
        feats[f'mean_{ph[-1]}'] = df[ph].mean()
    rms_vals = [feats['RMS_R'], feats['RMS_S'], feats['RMS_T']]
    feats['amp_imbalance'] = (max(rms_vals) - min(rms_vals)) / np.mean(rms_vals)

    diff_RS = df['current_R'] - df['current_S']
    diff_ST = df['current_S'] - df['current_T']
    diff_TR = df['current_T'] - df['current_R']
    feats['std_diff_RS'] = diff_RS.std()
    feats['std_diff_ST'] = diff_ST.std()
    feats['std_diff_TR'] = diff_TR.std()

    feats['corr_RS'] = np.corrcoef(df['current_R'], df['current_S'])[0, 1]
    feats['corr_ST'] = np.corrcoef(df['current_S'], df['current_T'])[0, 1]
    feats['corr_TR'] = np.corrcoef(df['current_T'], df['current_R'])[0, 1]
    return feats

records = []
for dataset_id in sorted(os.listdir(BASE_DIR), key=lambda x: int(x) if x.isdigit() else x):
    ds_path = os.path.join(BASE_DIR, dataset_id)
    if not os.path.isdir(ds_path): 
        continue
    for f in sorted(os.listdir(ds_path)):
        if not f.endswith(".csv"):
            continue
        p = os.path.join(ds_path, f)
        try:
            df = pd.read_csv(p)
            if not {"current_R","current_S","current_T"}.issubset(df.columns):
                continue
            feats = extract_features(df)
            feats["dataset"] = int(dataset_id)
            feats["window"]  = f
            feats["path"]    = p
            records.append(feats)
        except Exception as e:
            print("Ошибка", p, e)

df_feat = pd.DataFrame(records)

# Нормировка и IsolationForest
df_z = df_feat.drop(columns=["dataset","window","path"]).apply(zscore)
iso  = IsolationForest(contamination=0.03, random_state=42)
df_feat["iso_label"] = iso.fit_predict(df_z)
df_feat["iso_score"] = iso.decision_function(df_z)

df_feat["manual_score"] = (
    df_z["amp_imbalance"].abs()*0.4 +
    df_z["std_diff_RS"].abs()*0.3 +
    df_z["std_diff_ST"].abs()*0.15 +
    df_z["std_diff_TR"].abs()*0.15
)
# ====== КОНЕЦ ВЫЧИСЛЕНИЯ ПРИЗНАКОВ ======


TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:
# ====== 1. ВЫБОР ГРУПП ======
df_sorted = df_feat.sort_values(SCORE_COL, ascending=False)  # по убыванию: аномальные сверху

anom = df_sorted.head(ANOMALY_N).copy()
norm = df_sorted.tail(NORMAL_N).copy()   # самые маленькие значения — норма

print("Топ аномалий:")
display(anom[["dataset","window",SCORE_COL,"iso_score"]])

print("Контрольная группа нормы:")
display(norm[["dataset","window",SCORE_COL,"iso_score"]])
